# 0. ReadMe

**LangChain과 Chroma를 활용한 RAG 구성**
- **0. LLM의 Knowledge Base가 될 데이터 준비**
  - 국가법령정보센터 소득세법(docx)을 다운받고 프로젝트 워킹 디렉토리에 옮깁니다.
- **1. 데이터 생성 및 분할**: 
  - `RecursiveCharacterTextSplitter`를 사용하여 문서를 chunk로 분할하고, `Docx2txtLoader`를 통해 데이터를 로드합니다.
- **2. 데이터 임베딩 및 저장**: 
  - OpenAI의 임베딩 모델을 사용하여 chunk를 벡터화하고, `Chroma`를 통해 벡터화된 데이터를 데이터베이스에 저장합니다.
- **3. 질의 응답 생성**: 
  - 저장된 데이터를 유사도 검색을 통해 검색하고, `RetrievalQA` 체인을 사용하여 질문에 대한 답변을 생성합니다.

# 1. 필요 라이브러리 불러오기

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain import hub
from langchain.chains import RetrievalQA
# 환경변수 불러오기
from dotenv import load_dotenv
load_dotenv()

True

# 2. Knowledge Base 구성을 위한 데이터 생성

- [RecursiveCharacterTextSplitter](https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/)를 활용한 데이터 chunking
    - split 된 데이터 chunk를 Large Language Model(LLM)에게 전달하면 토큰 절약 가능
    - 비용 감소와 답변 생성시간 감소의 효과
    - LangChain에서 다양한 [TextSplitter](https://python.langchain.com/v0.2/docs/how_to/#text-splitters)들을 제공
- `chunk_size` 는 split 된 chunk의 최대 크기
- `chunk_overlap`은 앞 뒤로 나뉘어진 chunk들이 얼마나 겹쳐도 되는지 지정

In [42]:
# from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax_docs/tax.docx')
loader

In [43]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)
document_list = loader.load_and_split(text_splitter=text_splitter)
len(document_list)

189

In [75]:
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings


# OpenAI에서 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [71]:
from langchain_chroma import Chroma

# 데이터를 처음 저장할 때 
database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax', persist_directory="./chroma")

# 이미 저장된 데이터를 사용할 때 
# database = Chroma(collection_name='chroma-tax', persist_directory="./chroma", embedding_function=embedding)

GoogleGenerativeAIError: Error embedding content: 400 * BatchEmbedContentsRequest.model: unexpected model name format
* BatchEmbedContentsRequest.requests[0].model: unexpected model name format
* BatchEmbedContentsRequest.requests[1].model: unexpected model name format
* BatchEmbedContentsRequest.requests[2].model: unexpected model name format
* BatchEmbedContentsRequest.requests[3].model: unexpected model name format
* BatchEmbedContentsRequest.requests[4].model: unexpected model name format
* BatchEmbedContentsRequest.requests[5].model: unexpected model name format
* BatchEmbedContentsRequest.requests[6].model: unexpected model name format
* BatchEmbedContentsRequest.requests[7].model: unexpected model name format
* BatchEmbedContentsRequest.requests[8].model: unexpected model name format
* BatchEmbedContentsRequest.requests[9].model: unexpected model name format
* BatchEmbedContentsRequest.requests[10].model: unexpected model name format
* BatchEmbedContentsRequest.requests[11].mode

In [ ]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# 현재 API 키로 사용 가능한 모든 임베딩 모델 출력

for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [64]:
# from dotenv import load_dotenv
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
      print("\n####", m.name, "####")
      
      try:
        print(f"임베딩: ", end="")
        embedding = GoogleGenerativeAIEmbeddings(model= m.name)
        print(f"✅ 성공!")
      except Exception as e:
        print(f"❌ 실패!")
        print(f"발생한 에러:", end="")
        print(f"   {type(e).__name__}: {e}")
      
      try:
        print(f"크로마 저장:", end="")
        database = Chroma.from_documents(
        documents=document_list, 
        embedding=embedding, 
        collection_name='chroma-tax', 
        persist_directory="./chroma"
      )
        print(f"✅ 성공!")
      except Exception as e:
        print(f"❌ 실패!")
        print(f"발생한 에러:", end="")
        print(f"   {type(e).__name__}: {e}")



#### models/gemini-embedding-001 ####
임베딩: ✅ 성공!
크로마 저장:❌ 실패!
발생한 에러:   GoogleGenerativeAIError: Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit.  [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
]

#### models/gemini-embedding-2-preview ####
임베딩: ✅ 성공!
크로마 저장:❌ 실패!
발생한 에러:   GoogleGenerativeAIError: Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit.  [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
]

#### mode

In [ ]:
# from langchain_chroma import Chroma
database = Chroma(
      embedding_function=embedding,
      # ⭐️ 크로마는 램에 저장되기 때문에 
      # ⭐️ 디스크에 저장하기 위해서는 폴더 지정! 
      # ⭐️ persist_directory 옵션 사용
      persist_directory="./chroma-directory",   # 저장할 폴더이름
      collection_name="chroma-tax-table",       # 그 안의 테이블/서랍 이름 # 디폴트값은 "langchain" 
)

In [79]:
import time

def add_documents_with_retry(database, documents, max_retries=5):
    for attempt in range(max_retries): # [0, 1, 2, 3, 4]
        try:
            database.add_documents(documents) # DB에 문서를 추가로 넣는 함수
            return True # 정상적으로 넣었으면 True 반환

        except Exception as e: # add_documents 중 예외가 나면 # 메시지 문자열로 바꿈
            error_message = str(e) 
            # 429 또는 RESOURCE_EXHAUSTED 메시지가 아닌 에러라면 raise (에러를 그대로 발생시킴, 실행 중단)
            if (
                "429" not in error_message
                and "RESOURCE_EXHAUSTED" not in error_message
            ):
                raise
            # 429 또는 RESOURCE_EXHAUSTED 메시지가 나오면 잠시 대기 후 재시도
            else:
              wait_seconds = (2 ** attempt) * 10
              print(f"사용량 제한 발생, {wait_seconds}초 후 재시도")
              time.sleep(wait_seconds)  # 슬립한다음에 최대 5번 반복

    return False

In [81]:
batch_size = 5

for start in range(0, len(document_list), batch_size): # 0~312까지 5씩 증가한 리스트 [0, 5, ..., 310]]
    batch = document_list[start:start + batch_size] # document_chunk_list[0:5], [5:10], ..., [310:315]

    # 문서를 넣는 함수
    success = add_documents_with_retry(
        database=database,
        documents=batch  # documents는 5개청크씩 분할해서 넣음!
    )

    if not success:
        print(f"{start}번째 청크부터 저장 실패")
        break

    print(
        f"{start + 1}번 ~ "
        f"{start + len(batch)}번 청크 저장 완료"
    )

    time.sleep(5)

1번 ~ 5번 청크 저장 완료
6번 ~ 10번 청크 저장 완료
11번 ~ 15번 청크 저장 완료
16번 ~ 20번 청크 저장 완료
21번 ~ 25번 청크 저장 완료
26번 ~ 30번 청크 저장 완료
31번 ~ 35번 청크 저장 완료
36번 ~ 40번 청크 저장 완료
사용량 제한 발생, 10초 후 재시도
사용량 제한 발생, 20초 후 재시도
41번 ~ 45번 청크 저장 완료
46번 ~ 50번 청크 저장 완료
51번 ~ 55번 청크 저장 완료
56번 ~ 60번 청크 저장 완료
61번 ~ 65번 청크 저장 완료
사용량 제한 발생, 10초 후 재시도
사용량 제한 발생, 20초 후 재시도
66번 ~ 70번 청크 저장 완료
71번 ~ 75번 청크 저장 완료
76번 ~ 80번 청크 저장 완료
81번 ~ 85번 청크 저장 완료
86번 ~ 90번 청크 저장 완료
사용량 제한 발생, 10초 후 재시도
사용량 제한 발생, 20초 후 재시도
91번 ~ 95번 청크 저장 완료
96번 ~ 100번 청크 저장 완료
101번 ~ 105번 청크 저장 완료
106번 ~ 110번 청크 저장 완료
111번 ~ 115번 청크 저장 완료
사용량 제한 발생, 10초 후 재시도
사용량 제한 발생, 20초 후 재시도
116번 ~ 120번 청크 저장 완료
121번 ~ 125번 청크 저장 완료
126번 ~ 130번 청크 저장 완료
131번 ~ 135번 청크 저장 완료
136번 ~ 140번 청크 저장 완료
사용량 제한 발생, 10초 후 재시도
사용량 제한 발생, 20초 후 재시도
141번 ~ 145번 청크 저장 완료
146번 ~ 150번 청크 저장 완료
151번 ~ 155번 청크 저장 완료
156번 ~ 160번 청크 저장 완료
161번 ~ 165번 청크 저장 완료
사용량 제한 발생, 10초 후 재시도
사용량 제한 발생, 20초 후 재시도
사용량 제한 발생, 40초 후 재시도
166번 ~ 170번 청크 저장 완료
171번 ~ 175번 청크 저장 완료
176번 ~ 180번 청크 저장 완료
181번 ~ 185번 청

# 3. 답변 생성을 위한 Retrieval

- `Chroma`에 저장한 데이터를 유사도 검색(`similarity_search()`)를 활용해서 가져옴

In [85]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'

# `k` 값을 조절해서 얼마나 많은 데이터를 불러올지 결정
retrieved_docs = database.similarity_search(query, k=3)
retrieved_docs

[Document(metadata={'source': './tax_docs/tax.docx'}, page_content='제5관 근로소득공제ㆍ연금소득공제 및 퇴직소득공제 <개정 2009. 12. 31.>\n\n\n\n제47조(근로소득공제) ①근로소득이 있는 거주자에 대해서는 해당 과세기간에 받는 총급여액에서 다음의 금액을 공제한다. 다만, 공제액이 2천만원을 초과하는 경우에는 2천만원을 공제한다. <개정 2012. 1. 1., 2014. 1. 1., 2019. 12. 31.>\n\n\n\n② 일용근로자에 대한 공제액은 제1항에도 불구하고 1일 15만원으로 한다.<개정 2018. 12. 31.>\n\n③ 근로소득이 있는 거주자의 해당 과세기간의 총급여액이 제1항 또는 제2항의 공제액에 미달하는 경우에는 그 총급여액을 공제액으로 한다.\n\n④ 제1항부터 제3항까지의 규정에 따른 공제를 “근로소득공제”라 한다.\n\n⑤ 제1항의 경우에 2인 이상으로부터 근로소득을 받는 사람(일용근로자는 제외한다)에 대하여는 그 근로소득의 합계액을 총급여액으로 하여 제1항에 따라 계산한 근로소득공제액을 총급여액에서 공제한다.<개정 2010. 12. 27.>\n\n⑥ 삭제<2010. 12. 27.>\n\n[전문개정 2009. 12. 31.]\n\n\n\n제47조의2(연금소득공제) ①연금소득이 있는 거주자에 대해서는 해당 과세기간에 받은 총연금액(분리과세연금소득은 제외하며, 이하 이 항에서 같다)에서 다음 표에 규정된 금액을 공제한다. 다만, 공제액이 900만원을 초과하는 경우에는 900만원을 공제한다. <개정 2013. 1. 1.>\n\n\n\n② 제1항에 따른 공제를 “연금소득공제”라 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제48조(퇴직소득공제) ① 퇴직소득이 있는 거주자에 대해서는 해당 과세기간의 퇴직소득금액에서 제1호의 구분에 따른 금액을 공제하고, 그 금액을 근속연수(1년 미만의 기간이 있는 경우에는 이를 1년으로 보며, 제22조제1항제1호의 경우에는 

# 4. Augmentation을 위한 Prompt 활용

- Retrieval된 데이터는 LangChain에서 제공하는 프롬프트(`"rlm/rag-prompt"`) 사용

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash')

In [98]:
# 2. 기존 프롬프트  보완
prompt = f"""
[Identity]
- 당신은 최고의 한국 소득세 전문가 입니다
- [Context]를 참고해서 사용자의 질문에 답변을 주세요

[Context]
{retrieved_docs}

[Question]
{query}
"""

In [101]:
llm.invoke(prompt)

AIMessage(content='제공해주신 법령 조문(Context)을 바탕으로 답변드립니다.\n\n제시된 자료만으로는 연봉 5,000만 원인 직장인의 **정확한 소득세 액수를 산출할 수 없습니다.** 그 이유는 다음과 같습니다.\n\n1. **근로소득공제액 미비**: 제47조(근로소득공제)에 따라 총급여액 5,000만 원에 대한 공제액을 차감해야 하나, 상세 공제율표가 본문 내용에서 생략되어 있습니다.\n2. **세율 미비**: 제55조(세율)에 따라 근로소득금액에서 종합소득공제를 차감한 과세표준에 적용할 구체적인 소득세율 구간(표)이 생략되어 있습니다.\n3. **개인별 공제 정보 부족**: 제50조(기본공제)에 따른 본인(150만 원) 외 부양가족 수, 제51조(추가공제) 및 기타 세액공제 적용 여부에 대한 구체적인 정보가 필요합니다.\n\n---\n\n### Context에 따른 기본 계산 절차\n법령에 명시된 소득세 계산 흐름은 다음과 같습니다.\n\n1. **근로소득금액 산출**: 총급여액(5,000만 원) - **근로소득공제** (제47조)\n2. **과세표준 산출**: 근로소득금액 - **종합소득공제** (제50조 기본공제 1인당 150만 원 등)\n3. **산출세액 계산**: 과세표준 × **세율** (제55조)\n4. **최종 납부세액**: 산출세액 - **세액공제/감면**\n\n정확한 세액을 계산하려면 **구체적인 근로소득공제표, 소득세율표, 부양가족 수 및 기타 공제 내역**이 구비되어야 합니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-27f735b9-ef1b-4098-bfcf-e10a35f9da08-0', usage_metadata={'input_tokens': 3223, 'output_to

# 5. 답변 생성

- [RetrievalQA](https://docs.smith.langchain.com/old/cookbook/hub-examples/retrieval-qa-chain)를 통해 LLM에 전달
    - `RetrievalQA`는 [create_retrieval_chain](https://python.langchain.com/v0.2/docs/how_to/qa_sources/#using-create_retrieval_chain)으로 대체됨
    - 실제 ChatBot 구현 시 `create_retrieval_chain`으로 변경하는 과정을 볼 수 있음

In [ ]:
# from langchain import hub

prompt = hub.pull("rlm/rag-prompt")
print(prompt.messages[0].prompt.template)

/Users/jeong-yujin/.pyenv/versions/inflearn-llm-application-ver2/lib/python3.10/site-packages/langsmith/client.py:354: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:


In [ ]:
# from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm, 
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

In [94]:
ai_message = qa_chain({"query": query})

/var/folders/j3/d7fn7bf56qdc7k_w705hspf80000gn/T/ipykernel_93099/3455095564.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  ai_message = qa_chain({"query": query})


In [95]:
# 강의에서는 위처럼 진행하지만 업데이트된 LangChain 문법은 `.invoke()` 활용을 권장
ai_message = qa_chain.invoke({"query": query})

In [96]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '제공된 문서에는 연봉 5천만원에 적용되는 구체적인 근로소득공제액 계산 기준과 종합소득세율 정보가 포함되어 있지 않습니다. 따라서 제공된 정보만으로는 연봉 5천만원인 직장인의 정확한 소득세를 알 수 없습니다.'}